# 👕 Semana 09: Fashion MNIST - Clustering No Supervisado
## Dataset: Fashion MNIST (Prendas de vestir)

**Objetivo:** Agrupar imágenes similares **sin usar las etiquetas** (clustering no supervisado).

**Algoritmos a competir:**
- **K-Means**: Clustering por centroides
- **DBSCAN**: Clustering por densidad (detecta outliers)
- **PCA**: Reducción a 2D/3D para visualizar clusters

**Métricas de evaluación:**
- Silhouette Score (qué tan bien separados están los clusters)
- Inercia (distancia intra-cluster - solo K-Means)
- Comparación visual con etiquetas reales (solo para validación)

**Contexto de negocio:** Un e-commerce quiere agrupar productos similares automáticamente para recomendar "productos relacionados".

---

### ¿Qué es?
Dataset de **70,000 imágenes** (28×28 píxeles en escala de grises) de **10 categorías de ropa**:
- Camiseta, Pantalón, Suéter, Vestido, Chaqueta
- Sandalias, Camisa, Zapatillas, Bolso, Botines

### ¿Qué problema resuelve?
**NO SUPERVISADO** - Agrupar imágenes similares **sin usar las etiquetas** (clustering).

### El Reto
**No usar las etiquetas** para entrenar. El algoritmo debe descubrir por sí mismo que las sandalias son similares entre sí, o que los pantalones forman un grupo.

### Contexto de negocio
Un e-commerce quiere agrupar productos similares automáticamente para recomendar "productos relacionados".

## 1. Configuración Inicial

Importamos las librerías necesarias y configuramos la semilla para reproducibilidad.

In [ ]:
# ======================================================
# SEMANA 09: FASHION MNIST - CLUSTERING NO SUPERVISADO
# ======================================================

# Instalar librerías necesarias
!pip install scikit-learn pandas numpy matplotlib seaborn -q

# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score, silhouette_samples

# Configuración
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("✅ Librerías importadas correctamente")

## 2. Carga y Exploración de Datos

Cargamos el dataset Fashion MNIST. Usaremos una muestra para tiempos razonables.

In [ ]:
print("="*60)
print("📊 CARGANDO DATASET FASHION MNIST")
print("="*60)

# Cargar Fashion MNIST (tarda unos segundos)
fashion_mnist = fetch_openml(data_id=40996, as_frame=False)
X = fashion_mnist.data
y = fashion_mnist.target.astype(int)

# Nombres de las clases
class_names = [
    'Camiseta', 'Pantalón', 'Suéter', 'Vestido', 'Chaqueta',
    'Sandalias', 'Camisa', 'Zapatillas', 'Bolso', 'Botines'
]

print(f"\n✅ Dataset cargado exitosamente")
print(f"📊 SHAPE: {X.shape[0]:,} muestras × {X.shape[1]} características (28×28 píxeles)")
print(f"🎯 Target: 10 categorías de ropa")

# Para tiempos razonables, usaremos una muestra
use_sample = True
sample_size = 10000  # 10,000 muestras es suficiente para clustering

if use_sample:
    from sklearn.model_selection import train_test_split
    _, X, _, y = train_test_split(X, y, train_size=sample_size/len(X), 
                                   random_state=42, stratify=y)
    print(f"\n⚠️ Usando muestra de {sample_size:,} muestras para rapidez")

# Normalizar los datos (escalar entre 0 y 1)
X = X / 255.0

# Mostrar algunas imágenes de ejemplo
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(28, 28), cmap='gray')
    ax.set_title(class_names[y[i]])
    ax.axis('off')
plt.suptitle('Ejemplos de Fashion MNIST', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n📊 Distribución de clases en la muestra:")
class_counts = pd.Series(y).value_counts().sort_index()
for i, count in class_counts.items():
    print(f"   • {class_names[i]}: {count} muestras ({count/len(y)*100:.1f}%)")

## 3. Reducción de Dimensionalidad con PCA

PCA reduce la dimensionalidad para visualización y mejora la eficiencia del clustering.

In [ ]:
print("="*60)
print("📊 REDUCCIÓN CON PCA")
print("="*60)

# Aplicar PCA para reducir a 50 componentes (retiene ~90% de varianza)
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)

# Varianza explicada
cumsum_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cumsum_var)+1), cumsum_var, 'bo-', linewidth=2)
plt.axhline(y=0.9, color='r', linestyle='--', label='90% varianza')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% varianza')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza acumulada')
plt.title('Varianza Explicada por PCA')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\n📊 Con 50 componentes se retiene el {cumsum_var[49]*100:.1f}% de la varianza")
print(f"📊 Shape original: {X.shape}")
print(f"📊 Shape después de PCA: {X_pca.shape}")

# PCA a 2D para visualización
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Clase real')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA 2D - Datos originales (coloreados por clase real)')

plt.subplot(1, 2, 2)
plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], alpha=0.6, s=5, color='steelblue')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA 2D - Datos originales (sin etiquetas)')

plt.tight_layout()
plt.show()

print("\n💡 PCA reduce 784 dimensiones a 50, manteniendo ~90% de la información.")

## 4. K-Means Clustering

K-Means agrupa los datos en k clusters basados en la distancia a los centroides.

In [ ]:
print("="*60)
print("📊 K-MEANS CLUSTERING")
print("="*60)

# Método del codo para encontrar el k óptimo
inertias = []
silhouettes = []
k_range = range(2, 15)

print("\n🔍 Calculando métricas para diferentes valores de k...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_pca)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_pca, kmeans.labels_)
    silhouettes.append(sil_score)
    print(f"   k={k}: Inercia={kmeans.inertia_:.0f}, Silhouette={sil_score:.4f}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo - K-Means')
axes[0].grid(True)

axes[1].plot(k_range, silhouettes, 'ro-', linewidth=2)
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score - K-Means (mayor es mejor)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Mejor k según silhouette
best_k = k_range[np.argmax(silhouettes)]
print(f"\n✅ Mejor k según Silhouette Score: {best_k}")

# Entrenar K-Means con el mejor k
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)

final_silhouette = silhouette_score(X_pca, kmeans_labels)
print(f"\n📊 Resultados de K-Means (k={best_k}):")
print(f"   • Inercia: {kmeans.inertia_:.0f}")
print(f"   • Silhouette Score: {final_silhouette:.4f}")

# Comparación con etiquetas reales (solo para evaluación)
ari = adjusted_rand_score(y, kmeans_labels)
nmi = normalized_mutual_info_score(y, kmeans_labels)
print(f"   • Adjusted Rand Index (vs real): {ari:.4f}")
print(f"   • Normalized Mutual Info (vs real): {nmi:.4f}")

## 5. DBSCAN Clustering

DBSCAN agrupa puntos basados en densidad, detectando automáticamente outliers.

In [ ]:
print("="*60)
print("📊 DBSCAN CLUSTERING")
print("="*60)

# Probar diferentes parámetros para DBSCAN
eps_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
min_samples_values = [5, 10, 15, 20]

print("\n🔍 Probando diferentes parámetros...")
best_score = -1
best_params = None
best_n_clusters = 0
best_noise = 0
best_dbscan_labels = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
        labels = dbscan.fit_predict(X_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        if n_clusters > 1 and n_clusters < 20:
            try:
                score = silhouette_score(X_pca[labels != -1], labels[labels != -1])
                if score > best_score:
                    best_score = score
                    best_params = (eps, min_samples)
                    best_n_clusters = n_clusters
                    best_noise = n_noise
                    best_dbscan_labels = labels
                print(f"   eps={eps}, min_samples={min_samples}: clusters={n_clusters}, "
                      f"ruido={n_noise} ({n_noise/len(labels)*100:.1f}%), silhouette={score:.4f}")
            except:
                pass

if best_params:
    print(f"\n✅ Mejores parámetros: eps={best_params[0]}, min_samples={best_params[1]}")
    print(f"\n📊 Resultados de DBSCAN:")
    print(f"   • Número de clusters: {best_n_clusters}")
    print(f"   • Puntos como ruido: {best_noise} ({best_noise/len(X_pca)*100:.1f}%)")
    print(f"   • Silhouette Score: {best_score:.4f}")
else:
    print("   No se encontraron parámetros adecuados")
    best_dbscan_labels = np.full(len(X_pca), -1)

## 6. Visualización de Clusters

Visualizamos los clusters encontrados por K-Means y DBSCAN en 2D usando PCA.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Clases reales
axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', alpha=0.6, s=5)
axes[0].set_title('Clases Reales (para referencia)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# K-Means
axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.6, s=5)
axes[1].set_title(f'K-Means (k={best_k})')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

# DBSCAN
if best_params:
    colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))
    for i in range(best_n_clusters):
        mask = best_dbscan_labels == i
        axes[2].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=[colors[i]], alpha=0.6, s=5, label=f'Cluster {i}')
    noise_mask = best_dbscan_labels == -1
    axes[2].scatter(X_pca_2d[noise_mask, 0], X_pca_2d[noise_mask, 1], c='gray', alpha=0.3, s=5, label='Ruido')
    axes[2].set_title(f'DBSCAN (eps={best_params[0]}, min_samples={best_params[1]})')
else:
    axes[2].text(0.5, 0.5, 'DBSCAN no generó clusters válidos', 
                 ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('DBSCAN')

axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')

plt.tight_layout()
plt.show()

## 7. Comparación de Métodos

Resumen de los resultados de los diferentes algoritmos de clustering.

In [ ]:
# Crear tabla comparativa
comparison_data = []

# K-Means
comparison_data.append({
    'Algoritmo': 'K-Means',
    'Parámetros': f'k={best_k}',
    'N° Clusters': best_k,
    'Silhouette': silhouette_score(X_pca, kmeans_labels),
    'Ruido (%)': 0,
    'ARI (vs real)': adjusted_rand_score(y, kmeans_labels),
    'NMI (vs real)': normalized_mutual_info_score(y, kmeans_labels)
})

# DBSCAN
if best_params:
    comparison_data.append({
        'Algoritmo': 'DBSCAN',
        'Parámetros': f'eps={best_params[0]}, min_samples={best_params[1]}',
        'N° Clusters': best_n_clusters,
        'Silhouette': best_score,
        'Ruido (%)': best_noise/len(X_pca)*100,
        'ARI (vs real)': adjusted_rand_score(y, best_dbscan_labels),
        'NMI (vs real)': normalized_mutual_info_score(y, best_dbscan_labels)
    })

comparison_df = pd.DataFrame(comparison_data)

print("="*60)
print("📊 TABLA COMPARATIVA DE ALGORITMOS")
print("="*60)
display(comparison_df.round(4))

## 8. Análisis de Silhouette

El Silhouette Score mide qué tan similares son los puntos dentro de un cluster comparado con otros clusters.

In [ ]:
# Calcular silhouette para K-Means
silhouette_vals = silhouette_samples(X_pca, kmeans_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de silhouette
y_lower = 10
for i in range(best_k):
    ith_cluster_silhouette = silhouette_vals[kmeans_labels == i]
    ith_cluster_silhouette.sort()
    size_cluster = len(ith_cluster_silhouette)
    y_upper = y_lower + size_cluster
    
    axes[0].fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette,
                          alpha=0.7, label=f'Cluster {i}')
    y_lower = y_upper + 5

axes[0].axvline(x=silhouette_score(X_pca, kmeans_labels), color='red', linestyle='--',
                label=f'Promedio: {silhouette_score(X_pca, kmeans_labels):.3f}')
axes[0].set_xlabel('Coeficiente de Silhouette')
axes[0].set_ylabel('Cluster')
axes[0].set_title(f'Análisis de Silhouette - K-Means (k={best_k})')
axes[0].legend()
axes[0].grid(True)

# Distribución de silhouette por cluster
cluster_silhouettes = []
for i in range(best_k):
    cluster_silhouettes.append(silhouette_vals[kmeans_labels == i].mean())

axes[1].bar(range(best_k), cluster_silhouettes, color='steelblue')
axes[1].axhline(y=silhouette_score(X_pca, kmeans_labels), color='red', linestyle='--',
                label=f'Promedio global: {silhouette_score(X_pca, kmeans_labels):.3f}')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Silhouette promedio')
axes[1].set_title('Silhouette por Cluster')
axes[1].legend()
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

print("\n📖 INTERPRETACIÓN:")
print("   • Silhouette > 0.5: Buena separación entre clusters")
print("   • Silhouette entre 0.2-0.5: Estructura razonable")
print("   • Silhouette < 0.2: Clusters superpuestos")

## 9. Simulación - Agrupación de una Nueva Imagen

Simulamos la clasificación de una nueva imagen de ropa usando los clusters encontrados.

In [ ]:
print("="*60)
print("🔮 SIMULACIÓN: CLASIFICACIÓN DE NUEVA PRENDA")
print("="*60)

# Tomar una imagen aleatoria del dataset (sin usar su etiqueta real)
np.random.seed(42)
idx = np.random.randint(0, len(X))
nueva_imagen = X[idx:idx+1]
imagen_original = nueva_imagen.reshape(28, 28)
etiqueta_real = y[idx]

# Aplicar PCA
nueva_imagen_pca = pca.transform(nueva_imagen)

# Predecir cluster con K-Means
cluster_kmeans = kmeans.predict(nueva_imagen_pca)[0]

# Mostrar la imagen
plt.figure(figsize=(3, 3))
plt.imshow(imagen_original, cmap='gray')
plt.title(f'Prenda real: {class_names[etiqueta_real]}')
plt.axis('off')
plt.show()

print("\n🔮 Resultados del clustering (sin usar la etiqueta real):")
print("-" * 50)
print(f"   • K-Means:  Cluster {cluster_kmeans}")
if best_params:
    cluster_dbscan = best_dbscan_labels[idx]
    if cluster_dbscan == -1:
        print(f"   • DBSCAN:   Detectado como RUIDO (no pertenece a ningún cluster)")
    else:
        print(f"   • DBSCAN:   Cluster {cluster_dbscan}")

# Mostrar ejemplos del mismo cluster de K-Means
cluster_indices = np.where(kmeans_labels == cluster_kmeans)[0][:5]

print(f"\n📋 Ejemplos de prendas en el mismo cluster (K-Means cluster {cluster_kmeans}):")
fig, axes = plt.subplots(1, 5, figsize=(10, 3))
for i, idx_ex in enumerate(cluster_indices):
    axes[i].imshow(X[idx_ex].reshape(28, 28), cmap='gray')
    axes[i].set_title(f'Real: {class_names[y[idx_ex]]}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

print("\n📖 INTERPRETACIÓN:")
print("   • El clustering agrupa prendas visualmente similares")
print("   • Un e-commerce puede usar esto para recomendar productos similares")
print("   • K-Means encontró grupos que se aproximan a las categorías reales")

## 10. Conclusiones

**Resumen de resultados:**

1. **K-Means**:
   - Encontró clusters en los datos
   - Ventaja: Rápido, fácil de interpretar
   - Desventaja: Requiere conocer k, sensible a outliers

2. **DBSCAN**:
   - Detecta puntos como ruido
   - Ventaja: No requiere k, detecta outliers
   - Desventaja: Sensible a parámetros eps y min_samples

3. **PCA**:
   - Redujo 784 dimensiones a 50 manteniendo ~90% de varianza
   - Permitió visualización en 2D

**Métrica más importante (Silhouette Score):**
- Mide la cohesión intra-cluster y separación inter-cluster
- Valores > 0.5 indican buena estructura de clusters

**Contexto de negocio:**
- Un e-commerce puede usar K-Means para agrupar productos similares
- Los clusters permiten recomendar "productos relacionados" automáticamente
- DBSCAN puede identificar productos atípicos que merecen atención especial

**Próximos pasos:**
- Probar otros algoritmos (Agglomerative Clustering, Gaussian Mixture Models)
- Usar t-SNE o UMAP para mejor visualización
- Ajustar parámetros con técnicas como GridSearch para clustering

---
**Fin de la Semana 09 - Clustering No Supervisado**